# Few-Shot Image Classification with Prototypical Networks + XAI (Kaggle-Ready)

This notebook provides a complete, reproducible, and memory-efficient framework for **few-shot image classification** using **Prototypical Networks** in PyTorch with explainability via **Grad-CAM** and **Saliency Maps**.

---

## Problem Setup
- Dataset format:

```
dataset/
├── class_1/
├── class_2/
...
├── class_8/
```

- 8 classes, 160 images/class
- Split: 80% train, 10% val, 10% test (stratified)
- Episodic learning: N-way K-shot (default: 8-way 5-shot)


In [ ]:
# =============================
# 1) Environment Setup
# =============================
import os, random, math, copy, time, json
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy import stats

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)

In [ ]:
# =============================
# 2) Configuration
# =============================
@dataclass
class CFG:
    data_root: str = '/kaggle/input/your-dataset-folder/dataset'
    image_size: int = 224
    batch_workers: int = 2

    # Episodic params
    n_way: int = 8
    k_shot: int = 5
    q_query: int = 5

    episodes_per_epoch: int = 80
    val_episodes: int = 30
    test_episodes: int = 100

    backbone: str = 'resnet18'  # or 'efficientnet_b0'
    embedding_dim: int = 512
    metric: str = 'euclidean'  # 'euclidean' or 'cosine'

    lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 30
    early_stop_patience: int = 8

    mixed_precision: bool = True
    grad_clip: float = 1.0

cfg = CFG()
print(cfg)

In [ ]:
# =============================
# 3) Dataset Loading + Stratified Split
# =============================
def build_dataframe(data_root):
    rows = []
    classes = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))])
    for cls in classes:
        cdir = os.path.join(data_root, cls)
        for fn in os.listdir(cdir):
            fp = os.path.join(cdir, fn)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                rows.append({'path': fp, 'class_name': cls})
    df = pd.DataFrame(rows)
    class_to_idx = {c:i for i,c in enumerate(sorted(df['class_name'].unique()))}
    df['label'] = df['class_name'].map(class_to_idx)
    return df, class_to_idx

df, class_to_idx = build_dataframe(cfg.data_root)
idx_to_class = {v:k for k,v in class_to_idx.items()}

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

print('Total:', len(df), 'Train:', len(train_df), 'Val:', len(val_df), 'Test:', len(test_df))
print('Classes:', class_to_idx)

In [ ]:
# =============================
# 4) Transformations
# =============================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size+16, cfg.image_size+16)),
    transforms.RandomCrop(cfg.image_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [ ]:
# =============================
# 5) Episodic Dataset + Sampler
# =============================
class EpisodicDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.by_class = defaultdict(list)
        for i, r in self.df.iterrows():
            self.by_class[int(r.label)].append(i)

    def __len__(self):
        return len(self.df)

    def load_item(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row.path).convert('RGB')
        return self.transform(img), int(row.label)

def sample_episode(edset, n_way, k_shot, q_query):
    classes = random.sample(list(edset.by_class.keys()), n_way)
    support_x, support_y, query_x, query_y = [], [], [], []
    for epi_label, cls in enumerate(classes):
        idxs = random.sample(edset.by_class[cls], k_shot + q_query)
        s_idxs, q_idxs = idxs[:k_shot], idxs[k_shot:]
        for i in s_idxs:
            x, _ = edset.load_item(i)
            support_x.append(x); support_y.append(epi_label)
        for i in q_idxs:
            x, _ = edset.load_item(i)
            query_x.append(x); query_y.append(epi_label)

    return (torch.stack(support_x), torch.tensor(support_y),
            torch.stack(query_x), torch.tensor(query_y),
            classes)

train_set = EpisodicDataset(train_df, train_tfms)
val_set = EpisodicDataset(val_df, eval_tfms)
test_set = EpisodicDataset(test_df, eval_tfms)

In [ ]:
# =============================
# 6) Prototypical Network
# =============================
class ProtoNet(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding_dim = 512

    def encode(self, x):
        z = self.feature_extractor(x).flatten(1)
        return z

    def forward_episode(self, support_x, support_y, query_x, metric='euclidean'):
        z_support = self.encode(support_x)
        z_query = self.encode(query_x)

        n_way = support_y.unique().numel()
        prototypes = []
        for c in range(n_way):
            prototypes.append(z_support[support_y == c].mean(0))
        prototypes = torch.stack(prototypes)

        if metric == 'euclidean':
            dists = torch.cdist(z_query, prototypes, p=2)
            logits = -dists
        elif metric == 'cosine':
            zq = F.normalize(z_query, dim=1)
            zp = F.normalize(prototypes, dim=1)
            logits = zq @ zp.T
        else:
            raise ValueError('metric must be euclidean or cosine')

        return logits, prototypes, z_support, z_query

model = ProtoNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision and device.type == 'cuda'))

In [ ]:
# =============================
# 7) Train / Val / Test Loops
# =============================
def run_episodes(edset, episodes, train_mode=True):
    model.train(train_mode)
    losses, preds_all, trues_all, probs_all = [], [], [], []

    for _ in range(episodes):
        sx, sy, qx, qy, _ = sample_episode(edset, cfg.n_way, cfg.k_shot, cfg.q_query)
        sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.mixed_precision and device.type == 'cuda')):
            logits, _, _, _ = model.forward_episode(sx, sy, qx, metric=cfg.metric)
            loss = F.cross_entropy(logits, qy)

        if train_mode:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()

        probs = F.softmax(logits, dim=1).detach().cpu().numpy()
        preds = logits.argmax(1).detach().cpu().numpy()
        trues = qy.detach().cpu().numpy()

        losses.append(loss.item())
        preds_all.extend(preds.tolist())
        trues_all.extend(trues.tolist())
        probs_all.append(probs)

    probs_all = np.concatenate(probs_all, axis=0)
    acc = accuracy_score(trues_all, preds_all)
    prec, rec, f1, _ = precision_recall_fscore_support(trues_all, preds_all, average='macro', zero_division=0)
    return np.mean(losses), acc, prec, rec, f1, np.array(trues_all), np.array(preds_all), probs_all

def compute_ece(y_true, probs, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        m = (confidences >= bins[i]) & (confidences < bins[i+1])
        if m.any():
            ece += np.abs(correctness[m].mean() - confidences[m].mean()) * m.mean()
    return float(ece)

In [ ]:
# 8) Training with early stopping
best_acc=-1
best_state=None
history={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
pat=0
for epoch in range(1,cfg.max_epochs+1):
    tr_loss,tr_acc,*_ = run_episodes(train_set,cfg.episodes_per_epoch,True)
    va_loss,va_acc,*_ = run_episodes(val_set,cfg.val_episodes,False)
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc); history['val_acc'].append(va_acc)
    scheduler.step(va_acc)
    print(f"Epoch {epoch:02d}: tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | va_loss={va_loss:.4f} va_acc={va_acc:.4f}")
    if va_acc>best_acc:
        best_acc=va_acc; pat=0
        best_state=copy.deepcopy(model.state_dict())
        torch.save({'model':best_state,'cfg':cfg.__dict__},'best_protonet.pt')
    else:
        pat+=1
        if pat>=cfg.early_stop_patience:
            print('Early stopping.')
            break
model.load_state_dict(best_state)


In [ ]:
# 9) Test evaluation + ECE + confusion matrix
te_loss,te_acc,te_prec,te_rec,te_f1,y_true,y_pred,probs = run_episodes(test_set,cfg.test_episodes,False)
ece = compute_ece(y_true, probs)
print({'test_loss':te_loss,'acc':te_acc,'precision':te_prec,'recall':te_rec,'f1':te_f1,'ece':ece})

cm=confusion_matrix(y_true,y_pred,labels=list(range(cfg.n_way)))
plt.figure(figsize=(7,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.title('Episode-level Confusion Matrix')
plt.xlabel('Pred'); plt.ylabel('True'); plt.show()


In [ ]:
# 10) Multiple-run evaluation + t-test + CI
runs=5
accs=[]
for r in range(runs):
    set_seed(SEED+r)
    _,a,_,_,_,_,_,_ = run_episodes(test_set,30,False)
    accs.append(a)
accs=np.array(accs)
mean=accs.mean(); std=accs.std(ddof=1)
ci=stats.t.interval(0.95, df=runs-1, loc=mean, scale=std/np.sqrt(runs))
tv,pv=stats.ttest_1samp(accs,0.125) # random guess baseline for 8-way
print('accs=',accs,'mean=',mean,'95%CI=',ci,'t=',tv,'p=',pv)


In [ ]:
# 11) Embedding visualization (PCA/t-SNE)
model.eval()
embs=[]; labs=[]
for _ in range(10):
    sx,sy,qx,qy,_=sample_episode(test_set,cfg.n_way,cfg.k_shot,cfg.q_query)
    with torch.no_grad():
        z=model.encode(qx.to(device)).cpu().numpy()
    embs.append(z); labs.append(qy.numpy())
embs=np.concatenate(embs); labs=np.concatenate(labs)

pca=PCA(n_components=2,random_state=SEED).fit_transform(embs)
ts=TSNE(n_components=2,random_state=SEED,init='pca',learning_rate='auto').fit_transform(embs)
for arr,name in [(pca,'PCA'),(ts,'t-SNE')]:
    plt.figure(figsize=(6,5))
    for c in np.unique(labs):
        m=labs==c
        plt.scatter(arr[m,0],arr[m,1],s=12,label=f'class_{c}')
    plt.title(name+' of query embeddings'); plt.legend(); plt.show()


In [ ]:
# 12) XAI: Grad-CAM + Saliency + attribution sparsity
class GradCAM:
    def __init__(self, model, target_layer):
        self.model=model; self.target_layer=target_layer
        self.grad=None; self.act=None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,'act',o))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grad',go[0]))
    def __call__(self, x, class_idx):
        self.model.zero_grad(set_to_none=True)
        out=self.model.encode(x)
        score=out[:,class_idx].sum()
        score.backward(retain_graph=True)
        w=self.grad.mean(dim=(2,3),keepdim=True)
        cam=(w*self.act).sum(1,keepdim=True)
        cam=F.relu(cam)
        cam=F.interpolate(cam,size=(cfg.image_size,cfg.image_size),mode='bilinear',align_corners=False)
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam.detach().cpu().squeeze().numpy()

def saliency_map(x):
    x=x.clone().requires_grad_(True)
    z=model.encode(x)
    s=z.norm(dim=1).sum(); s.backward()
    g=x.grad.abs().max(dim=1)[0]
    g=(g-g.min())/(g.max()-g.min()+1e-8)
    return g.detach().cpu().squeeze().numpy()

def attribution_sparsity(attr,th=0.2):
    return float((attr<th).mean())

sx,sy,qx,qy,_=sample_episode(test_set,cfg.n_way,cfg.k_shot,cfg.q_query)
img=qx[0:1].to(device)
cam=GradCAM(model, model.feature_extractor[-1][-1].conv2)(img, class_idx=0)
sal=saliency_map(img)
print('Attribution sparsity (Grad-CAM):', attribution_sparsity(cam), 'Saliency:', attribution_sparsity(sal))

inp=qx[0].permute(1,2,0).numpy(); inp=(inp*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)).clip(0,1)
fig,ax=plt.subplots(1,3,figsize=(12,4))
ax[0].imshow(inp); ax[0].set_title('Original'); ax[0].axis('off')
ax[1].imshow(inp); ax[1].imshow(cam,cmap='jet',alpha=0.45); ax[1].set_title('Grad-CAM'); ax[1].axis('off')
ax[2].imshow(inp); ax[2].imshow(sal,cmap='magma',alpha=0.45); ax[2].set_title('Saliency'); ax[2].axis('off')
plt.tight_layout(); plt.show()


## Mathematical Formulation

Prototype for class \(k\):
\[
\mathbf{c}_k = rac{1}{|S_k|}\sum_{(\mathbf{x}_i,y_i)\in S_k} f_\phi(\mathbf{x}_i)
\]

Euclidean metric logits:
\[
\ell_k(\mathbf{x}) = -\|f_\phi(\mathbf{x})-\mathbf{c}_k\|_2
\]

Cosine metric logits:
\[
\ell_k(\mathbf{x}) = rac{f_\phi(\mathbf{x})^	op \mathbf{c}_k}{\|f_\phi(\mathbf{x})\|\|\mathbf{c}_k\|}
\]

Class probability:
\[
p(y=k|\mathbf{x}) = 	ext{softmax}(\ell_k)
\]

ECE:
\[
\mathrm{ECE}=\sum_{b=1}^{B}rac{|I_b|}{n}\left|\mathrm{acc}(I_b)-\mathrm{conf}(I_b)ight|
\]
